In [ ]:
%pip install langchain
%pip install langchain-community
%pip install sentence-transformers
%pip install faiss-cpu
%pip install pypdf
%pip install transformers
%pip install torch

In [ ]:
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-text-splitters

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

In [6]:
# Step 1 — Load PDF

loader = PyPDFLoader("sample_data/sample.pdf")

documents = loader.load()

print("Total Pages:", len(documents))

Total Pages: 32


In [7]:
print(documents[0].page_content)

In [8]:
# Step 2 — Text Chunking


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 180


In [9]:
print(docs[0].page_content)

Python Tutorial 
        
    i 
 
About the Tutorial 
Today, Python is one of the most popular programming languages. Although it is a general-
purpose language, it is used in various areas of applications such as Machine Learning, 
Artificial Intelligence, web development, IoT, and more. 
This Python tutorial has been written for the beginners to help them understand the basic 
to advanced concepts of Python Programming Language. After completing this tutorial,


In [19]:
# Step 3 — Prepare Text Chunks

texts = [doc.page_content for doc in docs]

print("Total Text Chunks:", len(texts))

Total Text Chunks: 180


In [20]:
# Step 4 — Import Libraries for Embeddings and Vector Search

from sklearn.feature_extraction.text import TfidfVectorizer
import faiss
import numpy as np

In [21]:
# Step 5 — Generate Embeddings

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(texts)

print(tfidf_matrix.shape)

(180, 1804)


In [22]:
# Step 6 — Convert Embeddings to NumPy Array

embeddings = tfidf_matrix.toarray().astype('float32')

print(embeddings.shape)

(180, 1804)


In [24]:
# Step 7 — Create FAISS Vector Database

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)


In [25]:
# Step 8 — User Query

query = "What is the document mainly about?"

In [26]:
# Step 9 — Convert Query into Embedding

query_vector = vectorizer.transform([query]).toarray().astype('float32')

In [27]:
# Step 10 — Semantic Retrieval  

k = 3

distances, indices = index.search(query_vector, k)

retrieved_chunks = []

for idx in indices[0]:

    retrieved_chunks.append(texts[idx])

    print("\nRetrieved Chunk:\n")
    print(texts[idx])


Retrieved Chunk:

Python Tutorial 
        
    vi 
 
Table of Contents 
About the Tutorial ............................................................................................................................................ i 
What is Python? ............................................................................................................................................... i

Retrieved Chunk:

 Explicit 
 Readable 
The Zen of Python 
The Zen of Python is about code that not only works, but is Pythonic. Pythonic code is 
readable, concise, and maintainable.

Retrieved Chunk:

What is Python? 
Python is currently one of the most widely used programming languages. It is an 
interpreted programming language that operates at a high level. When compared to other 
languages, the learning curve for Python is much lower, and it is also quite straightforward 
to use. 
Python is the programming language of choice for professionals working in fields such as 
Artificial Intel

In [35]:
# Final RAG Answer Generation

query = "What is the document mainly about?"

# Convert query into vector
query_vector = vectorizer.transform([query]).toarray().astype('float32')

# Retrieve top chunks
k = 3

distances, indices = index.search(query_vector, k)

retrieved_chunks = []

for idx in indices[0]:
    retrieved_chunks.append(texts[idx])

# Create final context
context = "\n".join(retrieved_chunks)

# Final Answer
print("QUESTION:\n")
print(query)

print("\nFINAL ANSWER:\n")
print(context[:2000])

QUESTION:

What is the document mainly about?

FINAL ANSWER:

Python Tutorial 
        
    vi 
 
Table of Contents 
About the Tutorial ............................................................................................................................................ i 
What is Python? ............................................................................................................................................... i
 Explicit 
 Readable 
The Zen of Python 
The Zen of Python is about code that not only works, but is Pythonic. Pythonic code is 
readable, concise, and maintainable.
What is Python? 
Python is currently one of the most widely used programming languages. It is an 
interpreted programming language that operates at a high level. When compared to other 
languages, the learning curve for Python is much lower, and it is also quite straightforward 
to use. 
Python is the programming language of choice for professionals working in fields such as 
Artificial 